# 06. Gradient Boosting & XGBoost - Titanic**Topic:** Train sklearn's GradientBoostingClassifier and XGBoost on the Titanic dataset. Tune hyperparameters and compare.**Dataset:** Titanic (seaborn).

In [ ]:
# Run this cell first if running locally. In Google Colab everything is pre-installed.# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snssns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)np.random.seed(42)

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCVfrom sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifierfrom sklearn.metrics import accuracy_score, roc_auc_score, classification_report# Try XGBoost if availabletry:    from xgboost import XGBClassifier    HAS_XGB = Trueexcept ImportError:    HAS_XGB = False    print("XGBoost not installed. Run: pip install xgboost")df = sns.load_dataset("titanic")df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()df["family_size"] = df["sibsp"] + df["parch"] + 1df["is_alone"]    = (df["family_size"] == 1).astype(int)df["age"].fillna(df["age"].median(), inplace=True)df["embarked"].fillna(df["embarked"].mode()[0], inplace=True)df = pd.get_dummies(df, columns=["sex", "embarked"], drop_first=True)X = df.drop(columns=["survived"])y = df["survived"]X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 1. Compare three models

In [ ]:
rf  = RandomForestClassifier(n_estimators=300, random_state=42).fit(X_train, y_train)gb  = GradientBoostingClassifier(random_state=42).fit(X_train, y_train)for name, m in [("Random Forest", rf), ("Gradient Boosting", gb)]:    pred  = m.predict(X_test)    proba = m.predict_proba(X_test)[:, 1]    print(f"{name:20s}  Acc = {accuracy_score(y_test, pred):.3f}   AUC = {roc_auc_score(y_test, proba):.3f}")if HAS_XGB:    xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,                        use_label_encoder=False, eval_metric="logloss", random_state=42)    xgb.fit(X_train, y_train)    pred  = xgb.predict(X_test)    proba = xgb.predict_proba(X_test)[:, 1]    print(f"{'XGBoost':20s}  Acc = {accuracy_score(y_test, pred):.3f}   AUC = {roc_auc_score(y_test, proba):.3f}")

## 2. Tune Gradient Boosting with Grid Search

In [ ]:
param_grid = {    "learning_rate": [0.05, 0.1, 0.2],    "n_estimators":  [100, 300],    "max_depth":     [2, 3, 4],}grid = GridSearchCV(    GradientBoostingClassifier(random_state=42),    param_grid,    cv=5,    scoring="roc_auc",    n_jobs=-1,)grid.fit(X_train, y_train)print("Best params:", grid.best_params_)print(f"Best CV AUC: {grid.best_score_:.3f}")best = grid.best_estimator_print(f"\nTest AUC: {roc_auc_score(y_test, best.predict_proba(X_test)[:, 1]):.3f}")

## 3. Feature importance

In [ ]:
imp = pd.Series(best.feature_importances_, index=X.columns).sort_values()plt.figure(figsize=(8, 5))imp.plot.barh(color="gray")plt.title("Gradient Boosting Feature Importance")plt.show()

## 4. Exercises1. **Try LightGBM** (`from lightgbm import LGBMClassifier`).2. **Use early stopping** with XGBoost (`early_stopping_rounds=20`, supply `eval_set`).3. **Tune subsample and colsample_bytree** for XGBoost.4. **Try a Kaggle competition dataset** (House Prices Advanced Regression for tabular boosting practice).